In [1]:
import glob
import numpy as np
import pandas as pd
import xarray as xr
from metpy.interpolate import log_interpolate_1d
from metpy.units import units
import matplotlib.pyplot as plt

time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

def anomaly_calculation(model, clim):
    ano_list = []
    for i in range(1,13):
        ano_list.append(model.sel(time=model.time[model.time.dt.month.isin(i)]) - clim.sel(month=i))
    return xr.concat(ano_list, dim='time').sortby('time')

def calculate_anomaly_and_save_netcdf(model, var):
    ens = []
    ds = []
    for path in glob.glob('/sto4/data/Intermediate/VolMIP/pinatubo-full/' + model + '/' + var + '*fixed_dates.nc'):
        ens.append(path.split('/')[-1].split('_')[4])
        ds.append(xr.open_dataset(path))
    data = xr.concat(ds, dim='ens')
    data['ens'] = ens
    # open climatology and prepare it for anomaly calculation
    clim = xr.open_dataset(glob.glob('/sto4/data/Intermediate/VolMIP/piControl/' + model + '/' + var + '_clim*.nc')[0])
    clim['time'] = [1,2,3,4,5,6,7,8,9,10,11,12]
    clim = clim.rename({'time': 'month'})

    # select only the pressure levels that are available in the data
    data = data.sel(plev=clim.plev.values)
    # calculate anomalies
    data['ano'] = anomaly_calculation(data[var], clim[var])
    data.to_netcdf('/sto4/data/Intermediate/VolMIP/pinatubo-full/' + model + '/' + var + '_anomaly_' + model + '.nc')
    return 

In [2]:
models = ['CanESM5', 'GISS-E2-1-G','MPI-ESM1-2-LR', 'MIROC-ES2L', 'UKESM1-0-LL']

In [ ]:
for model in models: 
    calculate_anomaly_and_save_netcdf(model, 'ta')
    calculate_anomaly_and_save_netcdf(model, 'ua')